# Transform Customers Data
1. Filter out invalid rows i.e rows with null customer_id or duplicate customer_id
2. check if there are any null values in customer_unique_id
3. standarize customer_city and customer_state by converting all the city names into lowercase and state names into uppercase
4. write the transformed data to the silver table

In [0]:
#Imports
from pyspark.sql.functions import col,lower,upper,trim

In [0]:
customers_df=spark.read.table("olist_catalog.bronze.customers")

### Step1 - Filter out invalid rows i.e rows with null customer_id or duplicate customer_id

In [0]:
customers_valid_df=(
    customers_df.filter(col("customer_id").isNotNull())
        .dropDuplicates(subset=["customer_id"])
)

### Step2 - check if there are any null values in customer_unique_id

In [0]:
display(customers_valid_df.filter(col("customer_unique_id").isNull()))

### Step3 - standarize customer_city and customer_state by converting all the city names into lowercase and state names into uppercase

In [0]:
customers_final_df = (
    customers_valid_df.withColumns({'customer_city':lower(trim(col("customer_city"))),
                                  'customer_state':upper(trim(col("customer_state")))})
)

### Step4 - write the transformed data to the silver table

In [0]:
(
    customers_final_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("olist_catalog.silver.customers")

)

In [0]:
%sql
select * from olist_catalog.silver.customers